In [ ]:
# Written by tools/build_notebooks.py -- do not edit. The bootstrap cell below
# compares this against the repo it clones and tells you if these cells are old.
CELLS_SRC = "kaggle_04_evaluate.py"
CELLS_SHA = "e632ed4e4da1377d"

# 04 — The final numbers

**Accelerator: GPU T4 ×2.** **Runtime: ~15 minutes.** Costs roughly **0.3 GPU-hours**.

## Open the test set twice, not twenty times

Every choice you have made so far — which pooling, which preset, how many epochs — was made
on the **dev** set. This notebook is the only thing that touches the **test** set. Run it when
you are finished, not while you are still deciding. A test set you tune against is just a
second dev set with a misleading name.

## Before you press Run

1. **+ Add Input → Notebook Output →** notebook 00 (the store and the recipes)
2. **+ Add Input → Notebook Output →** notebook 01 (the Tier A counting model)
3. **+ Add Input → Notebook Output →** notebook 02 (the counter checkpoint)
4. **+ Add Input → Notebook Output →** notebook 03 (the separator checkpoint)
5. **Settings → Accelerator → GPU T4 ×2**

Missing pieces are skipped rather than fatal — you can evaluate the counter alone, or the
separator alone, and add the other later.

## Bootstrap (this cell is identical in every notebook)

Three ways to get the code onto the Kaggle machine, tried in order:

1. **GitHub clone** — set `REPO_URL` below and turn *Internet* ON in the notebook
   settings panel (Settings → Internet → On). This is the recommended route.
2. **Repo-as-dataset** — upload this folder as a Kaggle Dataset called
   `speaker-count-separate-v1` and attach it. No internet needed. Use this if your
   account cannot enable internet (phone-verification is required for that).
3. **Already there** — an existing clone is **fast-forwarded to the newest commit**,
   not reused as-is. A Kaggle session outlives many pushes, and silently running code
   from an hour ago is the most expensive kind of confusion: the log looks fine and the
   fix you are testing is not in it. Any local edits inside the clone are discarded.

Whichever route runs, the commit is printed. Every log can then be traced to the exact
code that produced it.

In [ ]:
REPO_URL = "https://github.com/AlAminAshraf01/speaker-count-separate-v1.git"
REPO_DIR = "/kaggle/working/speaker-count-separate-v1"
REPO_AS_DATASET = "/kaggle/input/speaker-count-separate-v1"

import hashlib
import os
import shutil
import subprocess
import sys


def cells_fingerprint(src_dir: str, name: str) -> str:
    """Short hash of one notebook's percent source plus this shared bootstrap.

    ``tools/build_notebooks.py`` stamps this into every generated ``.ipynb``. The copy
    running on Kaggle recomputes it from the freshly-cloned repo, so a notebook whose
    cells were imported before the last push says so in the first ten seconds instead of
    eleven hours later.

    Line endings are normalised first. The same file is CRLF in a Windows working tree
    and LF in a Linux clone, and a fingerprint that disagrees with itself across
    platforms is worse than no fingerprint at all.
    """
    digest = hashlib.sha256()
    for part in (name, "_bootstrap.py"):
        with open(os.path.join(src_dir, part), "rb") as fh:
            digest.update(fh.read().replace(b"\r\n", b"\n"))
        digest.update(b"\0")
    return digest.hexdigest()[:16]


def cells_status(repo_dir: str, src_name: str | None, stamp: str | None) -> str:
    """Compare the stamp baked into these cells with the repo they are about to run.

    Never raises. A check that can take down every notebook is a worse bug than the one
    it detects, so anything unreadable degrades to "cannot verify".
    """
    if not src_name or not stamp:
        return "unstamped -- re-import this notebook to enable the staleness check"
    try:
        current = cells_fingerprint(os.path.join(repo_dir, "notebooks", "src"), src_name)
    except Exception as exc:
        return f"cannot verify ({exc})"
    if current == stamp:
        return f"current ({stamp})"
    return "\n".join([
        f"STALE  cells {stamp} but repo has {current}",
        "",
        "  These notebook cells were imported before the newest push, so the fix you",
        "  are about to test is not in them. scripts/ and src/ just updated themselves;",
        "  notebook cells cannot, because Kaggle owns them.",
        "",
        "  Fix: File -> Import Notebook -> upload notebooks/" + src_name[:-3] + ".ipynb",
        "       again, re-attach the inputs, and re-run.",
    ])


def _git(repo_dir: str, *argv: str) -> subprocess.CompletedProcess:
    return subprocess.run(["git", "-C", repo_dir, *argv],
                          capture_output=True, text=True)


def update_clone(repo_dir: str) -> str:
    """Fast-forward an existing clone to the remote's newest commit.

    Returns a short status for printing; never raises. Losing internet is a reason to
    carry on with the code that is already there, but it is not a reason to be quiet
    about it -- running stale code unknowingly is how a fix gets tested without being
    present.
    """
    if not os.path.isdir(os.path.join(repo_dir, ".git")):
        return "not a git clone, left as it is"
    branch = _git(repo_dir, "rev-parse", "--abbrev-ref", "HEAD").stdout.strip() or "main"
    before = _git(repo_dir, "rev-parse", "--short", "HEAD").stdout.strip()
    fetched = _git(repo_dir, "fetch", "--depth", "1", "origin", branch)
    if fetched.returncode != 0:
        tail = (fetched.stderr or "").strip().splitlines()
        return f"COULD NOT FETCH ({tail[-1] if tail else 'unknown'}) -- code may be stale"
    reset = _git(repo_dir, "reset", "--hard", f"origin/{branch}")
    if reset.returncode != 0:
        tail = (reset.stderr or "").strip().splitlines()
        return f"COULD NOT UPDATE ({tail[-1] if tail else 'unknown'}) -- code may be stale"
    after = _git(repo_dir, "rev-parse", "--short", "HEAD").stdout.strip()
    return "already newest" if before == after else f"updated {before} -> {after}"


def describe_commit(repo_dir: str) -> str:
    """``<short sha> <date> <subject>`` for the checked-out commit, or a plain note."""
    out = _git(repo_dir, "log", "-1", "--format=%h %cs %s").stdout.strip()
    return out or "no git metadata"


def bootstrap(repo_url: str = REPO_URL, repo_dir: str = REPO_DIR) -> str:
    """Put the repo at `repo_dir`, put its `src/` on sys.path, and chdir into it."""
    if not os.path.isdir(os.path.join(repo_dir, "src")):
        if os.path.isdir(os.path.join(REPO_AS_DATASET, "src")):
            shutil.copytree(REPO_AS_DATASET, repo_dir, dirs_exist_ok=True)
            print(f"copied repo from the attached dataset {REPO_AS_DATASET}")
        else:
            subprocess.run(["git", "clone", "--depth", "1", repo_url, repo_dir], check=True)
            print(f"cloned {repo_url}")
    else:
        print(f"existing clone: {update_clone(repo_dir)}")
    src = os.path.join(repo_dir, "src")
    if src not in sys.path:
        sys.path.insert(0, src)
    os.chdir(repo_dir)
    return repo_dir


REPO = bootstrap()

import countsep  # noqa: E402

print("countsep", countsep.__version__, "at", REPO)
print("code ", describe_commit(REPO))
# CELLS_SRC / CELLS_SHA are set by the stamp cell that tools/build_notebooks.py puts at
# the top of every generated notebook. globals().get keeps this working in a notebook
# assembled by hand, where that cell may not exist.
CELLS = cells_status(REPO, globals().get("CELLS_SRC"), globals().get("CELLS_SHA"))
print("cells", CELLS if "\n" not in CELLS else "")
if "\n" in CELLS:
    print(CELLS)
print("python", sys.version.split()[0])

import torch  # noqa: E402

print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| devices", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  [{i}] {p.name}  {p.total_memory / 1e9:.1f} GB")

In [ ]:
import shlex
import time


def run(cmd: str, check: bool = True) -> int:
    """Run a shell command, streaming its output into the notebook."""
    print("$", cmd, flush=True)
    t0 = time.time()
    proc = subprocess.Popen(shlex.split(cmd), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    code = proc.wait()
    print(f"\n[exit {code} in {time.time() - t0:.1f}s]", flush=True)
    if check and code != 0:
        raise SystemExit(f"command failed with exit code {code}")
    return code

In [ ]:
import glob
import sys
import time
sys.path.insert(0, os.path.join(REPO, "scripts"))
from _common import autodetect_store, find_recipes

STORE = autodetect_store()
RECIPES_TEST = find_recipes("recipes_test.csv", STORE)

# Pin these to a specific path if you want to force a particular run; otherwise the
# newest attached checkpoint wins and every candidate is printed.
COUNTER_OVERRIDE = None
SEPARATOR_OVERRIDE = None


def _find(pattern, override=None):
    """The NEWEST matching checkpoint, with every candidate shown.

    This used to be `sorted(glob(...))[0]` -- alphabetical, silent. That is fine while one
    run is attached and quietly wrong the moment two are, which is exactly what happens
    when you re-run a training notebook and attach both versions to compare. Alphabetical
    order has nothing to do with which model you meant, and the final evaluation is the
    worst possible place to load the wrong weights without being told.
    """
    if override:
        print(f"    using the pinned path: {override}")
        return override
    hits = glob.glob(pattern, recursive=True)
    if not hits:
        return None
    hits.sort(key=os.path.getmtime, reverse=True)
    if len(hits) > 1:
        print(f"    {len(hits)} candidates matched {pattern} -- taking the newest:")
        for i, h in enumerate(hits):
            when = time.strftime("%Y-%m-%d %H:%M", time.localtime(os.path.getmtime(h)))
            print(f"      {'->' if i == 0 else '  '} {when}  {h}")
        print("      (set COUNTER_OVERRIDE / SEPARATOR_OVERRIDE above to pin one instead)")
    return hits[0]

COUNTER   = (_find("/kaggle/input/**/counter/ckpt/best.pt", COUNTER_OVERRIDE)
             or _find("/kaggle/input/**/ckpt/best.pt"))
SEPARATOR = _find("/kaggle/input/**/sep/ckpt/best.pt", SEPARATOR_OVERRIDE)
TIER_A    = _find("/kaggle/input/**/tier_a_model.joblib")

print("store       :", STORE)
print("recipes test:", RECIPES_TEST)
print("counter     :", COUNTER or "not attached -- counting will be skipped")
print("separator   :", SEPARATOR or "not attached -- separation will be skipped")
print("Tier A      :", TIER_A or "not attached")
if STORE is None or RECIPES_TEST is None:
    raise SystemExit("Attach notebook 00's output: '+ Add Input' -> 'Notebook Output'.")
if COUNTER is None and SEPARATOR is None:
    raise SystemExit("Attach notebook 02's and/or notebook 03's output -- nothing to evaluate.")

run(f"python scripts/preflight.py --for eval --store {STORE}"
    f" --recipes_test {RECIPES_TEST}"
    + (f" --ckpt {COUNTER}" if COUNTER else "")
    + f" --cells_src {CELLS_SRC} --cells_sha {CELLS_SHA}")

## Score the whole system  (~15 min)

### Four numbers, never one

A single score cannot describe a system that does two jobs, and averaging them hides which
half is broken. This prints all four:

**1. Counting accuracy and the full confusion matrix.** The classes are ordered, so guessing
4 when the answer is 3 is a much smaller mistake than guessing 1 — and plain accuracy treats
them identically. MAE and the matrix come with it.

**2. P-SI-SNR over the whole test set.** Ordinary SI-SDR is undefined when the predicted
count is wrong — there is no sensible pairing between 3 estimates and 4 true sources.
P-SI-SNR pads the mismatch with a −30 dB floor so the number stays defined. That matters
because it means a system cannot score well by quietly refusing to commit.

**3. SI-SDRi per N, count-correct clips only.** The fixed-N separation literature is always
*told* how many speakers there are, so this is the only row of yours that is comparable to it.

**4. SI-SDRi per N with the true count forced.** Separation quality given a perfect counter.
**The gap between 3 and 4 is exactly what miscounting costs you**, and it is the number that
tells you which half to work on next. Being able to ask that question is the practical payoff
of training the two models separately.

### And the confidence interval that is actually honest

Two are printed. The **Wilson** interval assumes the test clips are independent trials — they
are not, because the packer reserves 20 % of speakers for babble, so 1,500 clips come from
about **32 people**. The **speaker-level bootstrap** resamples people instead of clips. It is
wider, and it is the one to quote.

In [ ]:
cmd = (f"python scripts/06_evaluate.py"
       f" --store {STORE}"
       f" --recipes_test {RECIPES_TEST}"
       f" --n_boot 2000"
       f" --batch_size 16"
       f" --out /kaggle/working/eval")
if COUNTER:
    cmd += f" --counter {COUNTER}"
if SEPARATOR:
    cmd += f" --separator {SEPARATOR}"
if TIER_A:
    cmd += f" --tier_a {TIER_A}"
run(cmd)

## The table for your report

In [ ]:
import json

with open("/kaggle/working/eval/eval_report.json") as fh:
    report = json.load(fh)

print(f"frozen test set: {report['n']} mixtures from {report['n_speakers']} speakers\n")

if "counting" in report:
    c = report["counting"]
    lo, hi = c["speaker_bootstrap95"]
    print("COUNTING")
    print(f"  accuracy .............. {c['accuracy']:.1%}  [speaker 95 %: {lo:.1%}, {hi:.1%}]")
    print(f"  MAE ................... {c['mae']:.3f}")
    print(f"  majority-class naive .. {report['naive']['majority_class']['accuracy']:.1%}"
          f"   MAE {report['naive']['majority_class']['mae']:.3f}")

if "p_si_snr" in report:
    print("\nSEPARATION")
    print(f"  P-SI-SNR (all clips) .......... {report['p_si_snr']:+.2f} dB")
    for label, key in (("count-correct clips", "si_sdri_count_correct"),
                       ("true count forced  ", "si_sdri_oracle_count")):
        vals = report.get(key, {})
        if vals:
            row = "  ".join(f"N{n}:{v:+.1f}" for n, v in sorted(vals.items()))
            print(f"  SI-SDRi, {label} .. {row}")
    print(f"  published Conv-TasNet N=2 ..... +14.76 dB (200 epochs, train-360)")

## Opening the box  (~3 min, no training)

This is the course's interpretability deliverable, and it costs almost nothing: forward
passes on the checkpoints you already have. Protect this part when the schedule slips.

**The argument.** As N grows, the separator has to split the *same* learned filterbank among
more voices. If mask overlap rises and sparsity falls as N goes up, then the fact that
separation gets worse with more speakers has a *mechanical* explanation computed from the
network's own internals — rather than being a curve you point at and assert.

**And a question v0 could not ask.** In the old version the counting head sat on the
separator's own trunk, so any correlation between counting confidence and mask geometry was
partly just the two things being computed from the same tensor. Here the counter is a
completely separate model. If its confidence still correlates with the separator's mask
overlap, that says something about the **audio** — two independently trained models agreeing
that a particular mixture is hard — and not about a shared representation.

In [ ]:
if SEPARATOR:
    run(f"python scripts/07_interpret.py"
        f" --store {STORE}"
        f" --recipes_test {RECIPES_TEST}"
        f" --separator {SEPARATOR}"
        + (f" --counter {COUNTER}" if COUNTER else "")
        + f" --limit 300 --batch_size 8"
        f" --out /kaggle/working/interpret")
else:
    print("no separator attached -- skipping the mask-geometry analysis")

## Writing it up honestly

Four sentences worth putting in the report, because an examiner will ask and it is better to
have answered first.

1. **Quote the speaker-level interval and say which one it is.** A Wilson interval over 1,500
   clips drawn from 32 people understates the uncertainty.

2. **Say what "fully overlapped" means.** Every clip here has all N people talking at once for
   the full 3 seconds. Real conversation is mostly people taking turns, which is a different
   and harder problem (diarisation). A good number here does not mean speaker counting is
   solved.

3. **Report the gap to published separation as budget.** 14.76 dB comes from 200 epochs on
   train-clean-360. Give your epoch count and training set next to your number and the gap
   explains itself.

4. **The interesting finding is the comparison, not the absolute.** The old project put both
   jobs in one network; counting received 0.53 % of the gradient and scored at chance, and
   separation lost about 8.7 dB to the auxiliary objectives. Splitting them into two
   specialists and joining them at inference is the result — say what it cost (about 5 % more
   compute) and what it bought.

**Save Version → Save & Run All (Commit)** to keep the report.